# IBKR Flex sync

Pulls the "Trade History API" Flex Query (Cash Report + Open Positions + Trades) and brings `data/brokers/ibkr/` up to date. Safe to re-run: trades dedupe by transaction ID, so running this twice in a row just confirms nothing changed.

Run this regularly — the underlying Flex Query is scoped to "Last Business Day" on IBKR's side, so a missed run creates a permanent gap rather than something you can ask for later. If a run raises `TradeHistoryGapError`, see `docs/ibkr_flex_api.md` for how to backfill it. All the mechanics (protocol, error codes, why trades vs. snapshots are cached differently) are documented there too.

In [ ]:
from trades.brokers import ibkr
from trades.config import IbkrFlexApiConfig, IbkrFlexCredentials

credentials = IbkrFlexCredentials()  # reads IBKR_FLEX_WEB_SERVICE_TOKEN / IBKR_QUERY_ID from .env
config = IbkrFlexApiConfig()

## Run the sync

One network round trip (SendRequest, then poll GetStatement until ready), then trades/positions/cash are all updated from that single fetched statement.

In [ ]:
result = ibkr.sync_ibkr_account(credentials, config)
result

## What's in the cache now

In [ ]:
trades = ibkr.load_trade_history(config)
first_date = trades["trade_date"].min().date()
last_date = trades["trade_date"].max().date()
print(f"{len(trades)} trades cached, spanning {first_date} to {last_date}")
trades

In [ ]:
positions = ibkr.load_position_snapshots(config)
latest_positions = positions[positions["pulled_at"] == positions["pulled_at"].max()]
latest_positions

In [ ]:
cash = ibkr.load_cash_snapshots(config)
latest_cash = cash[cash["pulled_at"] == cash["pulled_at"].max()]
latest_cash